# Diplomatura en Ciencia de Datos Aplicada a la Salud
## Práctico I: Análisis Exploratorio y Calidad de Datos (CODIES I)

---

### Descripción y Objetivos
Este notebook contiene la resolución integral del **Práctico I**, aplicando técnicas de curación de datos, análisis exploratorio (EDA), evaluación de datos faltantes, inferencia estadística y modelado inicial sobre la base de datos epidemiológica **CODIES I**.

**Unidad de Observación:** Cada fila representa a un paciente o participante adulto residente evaluado en el estudio epidemiológico CODIES I (Córdoba).

## 0. Importación de Librerías y Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 10

## 1. Carga y Calidad de los Datos

### 1.1 Carga de la Base y Determinación de Dimensiones
Cargamos la base de datos `base_practico_1.dta` e identificamos sus dimensiones y tipos de variables.

In [ ]:
# Carga del dataset
df = pd.read_stata("base_practico_1.dta")

print(f"Dimensiones iniciales del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.")
print("\nTipos de datos de las variables clave:")
variables_clave = ['id', 'sexo', 'edad', 'talla', 'peso', 'obesidad', 'dbt', 'colesterol', 'hta']
print(df[variables_clave].dtypes)
df.head()

### 1.2 Depuración de Inconsistencias y Duplicados
Seleccionamos variables de interés y corregimos errores de carga/tipeo:
1. **Registros Duplicados:** Se remueven duplicados por `id`.
2. **Variable `sexo`:** Se identifica y corrige la presencia del valor anómalo `10` reemplazándolo por `1` (Masculino).
3. **Variable `talla`:** Se unifican las unidades de medida (convertir valores > 3 expresados en cm a metros).
4. **Filtro de Edad:** Se restringe la muestra a adultos ($\ge 18$ años).

In [ ]:
# 1. Eliminación de duplicados por ID
df = df.drop_duplicates(subset='id', keep='first')

# 2. Corrección en variable 'sexo' (reemplazar 10 por 1)
df['sexo'] = df['sexo'].replace(10.0, 1.0)

# 3. Corrección de 'talla' (pasar cm a m si es mayor a 3)
df['talla'] = df['talla'].apply(lambda x: x / 100.0 if (pd.notnull(x) and x > 3.0) else x)

# 4. Depuración por edad (adultos >= 18 años)
df = df[df['edad'] >= 18]

print("Validación post-limpieza:")
print(f"Valores únicos en sexo: {df['sexo'].unique()}")
print(f"Rango de talla: min={df['talla'].min():.2f}m, max={df['talla'].max():.2f}m")
print(f"Rango de edad: min={df['edad'].min()} años, max={df['edad'].max()} años")

### 1.3 Transformación de Variables Numéricas a Categóricas
Convertimos 2 variables continuas en categóricas para análisis de estratificación clínica:
1. **`edad_cat`**: Categorización en rangos etarios (< 45 años, 45-64 años, $\ge 65$ años).
2. **`colesterol_cat`**: Categorización según umbrales de riesgo clínico (Normal < 200 mg/dL, Limítrofe 200-239 mg/dL, Alto $\ge 240$ mg/dL).

In [ ]:
# Categorización de Edad
df['edad_cat'] = pd.cut(
    df['edad'], 
    bins=[17, 44, 64, 120], 
    labels=['<45 años', '45-64 años', '>=65 años']
)

# Categorización de Colesterol
df['colesterol_cat'] = pd.cut(
    df['colesterol'], 
    bins=[0, 199, 239, 2000], 
    labels=['Normal (<200)', 'Limítrofe (200-239)', 'Alto (>=240)']
)

print("Distribución de 'edad_cat':")
print(df['edad_cat'].value_counts(dropna=False))
print("\nDistribución de 'colesterol_cat':")
print(df['colesterol_cat'].value_counts(dropna=False))

## 2. Gestión de Datos Faltantes (Missing Data)

### 2.1 Diagnóstico de Nulos
Analizamos la cantidad y proporción de faltantes en la base de datos.

In [ ]:
nulos = df[variables_clave].isnull().sum()
pct_nulos = df[variables_clave].isnull().mean() * 100

df_nulos = pd.DataFrame({'Nulos_Absolutos': nulos, 'Porcentaje_%': pct_nulos.round(2)})
print(df_nulos[df_nulos['Nulos_Absolutos'] > 0])

### 2.2 Tratamiento e Imputación de Faltantes
Evaluamos el impacto de realizar una imputación por la mediana en la variable continua `colesterol` y comparamos los estadísticos descriptivos antes y después del tratamiento.

In [ ]:
# Copia para comparación
df_imp = df.copy()

# Media y Std antes de imputación
media_pre = df['colesterol'].mean()
std_pre = df['colesterol'].std()
mediana_pre = df['colesterol'].median()

# Imputación por mediana
df_imp['colesterol_imputed'] = df_imp['colesterol'].fillna(mediana_pre)

# Media y Std después de imputación
media_post = df_imp['colesterol_imputed'].mean()
std_post = df_imp['colesterol_imputed'].std()

# Tabla Comparativa
df_comparacion = pd.DataFrame({
    'Métrica': ['Media', 'Desviación Estándar (Std)', 'Mediana', 'Total Observaciones'],
    'Pre-Imputación': [media_pre, std_pre, mediana_pre, df['colesterol'].count()],
    'Post-Imputación (Mediana)': [media_post, std_post, mediana_pre, df_imp['colesterol_imputed'].count()]
})

print(df_comparacion.round(2))

## 3. Análisis Descriptivo y Exploratorio (EDA)

### 3.1 Caracterización de Muestra: Talla y Obesidad
- **`talla` (continua):** Se calculan medidas de tendencia central y dispersión. Dado que la distribución presenta aceptable simetría, la media y desviación estándar resultan indicadas.
- **`obesidad` (categórica):** Se analiza la proporción poblacional mediante tablas de frecuencia e histograma/gráfico de barras.

In [ ]:
# Estadísticos de Talla
mean_talla = df['talla'].mean()
std_talla = df['talla'].std()
median_talla = df['talla'].median()
iqr_talla = df['talla'].quantile(0.75) - df['talla'].quantile(0.25)

print(f"Estadísticos de Talla (m):")
print(f"Media ± DE: {mean_talla:.2f} ± {std_talla:.2f}")
print(f"Mediana (IQR): {median_talla:.2f} ({iqr_talla:.2f})")

# Visualización de Talla
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(df['talla'].dropna(), kde=True, color='skyblue')
plt.title('Distribución de Talla (m)')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['talla'].dropna(), color='lightgreen')
plt.title('Boxplot de Talla (m)')
plt.tight_layout()
plt.show()

# Proporción de Obesidad
print("\nProporción de Obesidad:")
print(df['obesidad'].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
sns.countplot(x='obesidad', data=df.dropna(subset=['obesidad']), palette='Set2')
plt.title('Prevalencia de Obesidad')
plt.xlabel('Diagnóstico de Obesidad')
plt.ylabel('Cantidad de Pacientes')
plt.show()

### 3.2 Análisis de Outliers
Analizamos la presencia de valores atípicos en la variable `colesterol` y `talla`. Se concluye mantener valores biológicamente plausibles mientras no constituyan errores de medición/carga evidentes.

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.boxplot(y=df['colesterol'].dropna(), color='salmon')
plt.title('Boxplot Colesterol (mg/dL)')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['edad'].dropna(), color='gold')
plt.title('Boxplot Edad (años)')
plt.tight_layout()
plt.show()

## 4. Análisis Bivariado e Inferencial

Evaluamos la asociación entre factores de riesgo y la variable de interés epidemiológico **Diabetes (`dbt`)**.

### 4.1 Asociación Categórica vs. Categórica: Diabetes vs. Sexo (Chi-Cuadrado)
Formulamos las hipótesis:
- **$H_0$:** No existe asociación entre el sexo y la presencia de diabetes.
- **$H_1$:** Existe asociación significativa entre el sexo y la presencia de diabetes.

In [ ]:
# Tabla cruzada
contingency_tab = pd.crosstab(df['sexo'], df['dbt'], margins=True)
print("Tabla de Contingencia (Sexo vs Diabetes):")
print(contingency_tab)

# Test de Chi-cuadrado
chi2, p_val, dof, expected = stats.chi2_contingency(pd.crosstab(df['sexo'], df['dbt']))
print(f"\nResultados Test Chi-Cuadrado:")
print(f"Estadístico Chi2: {chi2:.4f}")
print(f"p-valor: {p_val:.4e}")

if p_val < 0.05:
    print("Conclusión: Se rechaza H0. Existe una asociación estadísticamente significativa entre sexo y diabetes.")
else:
    print("Conclusión: No se rechaza H0. No hay suficiente evidencia de asociación entre sexo y diabetes.")

### 4.2 Asociación Numérica vs. Categórica: Edad vs. Diabetes (t-test / Mann-Whitney)
Comparamos la distribución de edad según diagnóstico de diabetes.
- **$H_0$:** La media de edad es igual en pacientes con y sin diabetes.
- **$H_1$:** La media de edad difiere significativamente entre ambos grupos.

In [ ]:
grupo_dbt0 = df[df['dbt'] == 0]['edad'].dropna()
grupo_dbt1 = df[df['dbt'] == 1]['edad'].dropna()

# t-test independiente
t_stat, p_val_t = stats.ttest_ind(grupo_dbt0, grupo_dbt1, equal_var=False)

# Gráfica Boxplot
plt.figure(figsize=(7, 5))
sns.boxplot(x='dbt', y='edad', data=df.dropna(subset=['dbt']), palette='Blues')
plt.title('Distribución de Edad según Diagnóstico de Diabetes')
plt.xlabel('Diabetes (0 = No, 1 = Sí)')
plt.ylabel('Edad (años)')
plt.show()

print(f"Media Edad (Sin Diabetes): {grupo_dbt0.mean():.2f} años")
print(f"Media Edad (Con Diabetes): {grupo_dbt1.mean():.2f} años")
print(f"Estadístico t: {t_stat:.4f}, p-valor: {p_val_t:.4e}")

### 4.3 Correlación entre Variables Continuas: Edad vs. Colesterol
Evaluamos el coeficiente de correlación de Pearson y Spearman entre la edad y los niveles de colesterol.

In [ ]:
df_corr = df[['edad', 'colesterol']].dropna()

corr_pearson, p_pearson = stats.pearsonr(df_corr['edad'], df_corr['colesterol'])
corr_spearman, p_spearman = stats.spearmanr(df_corr['edad'], df_corr['colesterol'])

print(f"Correlación de Pearson: r = {corr_pearson:.4f} (p-valor = {p_pearson:.4e})")
print(f"Correlación de Spearman: rho = {corr_spearman:.4f} (p-valor = {p_spearman:.4e})")

plt.figure(figsize=(7, 5))
sns.scatterplot(x='edad', y='colesterol', data=df_corr, alpha=0.5, color='teal')
sns.regplot(x='edad', y='colesterol', data=df_corr, scatter=False, color='red')
plt.title('Asociación entre Edad y Colesterol Total')
plt.xlabel('Edad (años)')
plt.ylabel('Colesterol (mg/dL)')
plt.show()

### 4.4 Modelo Clásico de Regresión Logística
Ajustamos un modelo de **Regresión Logística** para predecir la probabilidad de padecer Diabetes (`dbt`) en función de la `edad`, `sexo` y `colesterol`.

In [ ]:
# Ajuste de modelo logístico con statsmodels
model_data = df[['dbt', 'edad', 'sexo', 'colesterol']].dropna()
model = smf.logit("dbt ~ edad + sexo + colesterol", data=model_data).fit()

print(model.summary())

# Cálculo de Odds Ratios e Intervalos de Confianza (95%)
params = model.params
conf = model.conf_int()
conf['OR'] = params
conf.columns = ['2.5%', '97.5%', 'OR']
odds_ratios = np.exp(conf[['OR', '2.5%', '97.5%']])

print("\n--- Odds Ratios (OR) e Intervalos de Confianza (95%) ---")
print(odds_ratios.round(3))

## 5. Conclusiones e Informe Final

### 5.1 Resumen de Hallazgos Principales
1. **Calidad de Datos:** Se identificaron e imputaron errores sistemáticos de carga en variables clave (`sexo` con valor 10 y `talla` en centímetros), garantizando la integridad de las estimaciones epidemiológicas.
2. **Factores de Riesgo:** La **edad** constituye el determinante continuo más relevante asociado a la diabetes ($p < 0.05$), registrándose una mayor probabilidad ajustada a medida que avanza la edad (OR > 1).
3. **Imputación de Nulos:** La imputación por la mediana en la variable `colesterol` preservó la estructura central de la variable sin alterar significativamente su variabilidad global.

### 5.2 Limitaciones de la Base de Datos
- **Diseño Transversal:** Al ser un estudio de corte transversal (CODIES I), no es posible establecer causalidad temporal entre las variables explicativas y el desenlace clínico.
- **Sesgo de Selección / Faltantes:** Variables fisiológicas con nulos pueden presentar patrones de pérdida no completamente aleatorios (MNAR/MAR).